# **Thông tin nhóm**
- Lớp: ML 23KHDL1
- Nhóm: 6
- Sinh viên:
    - 23127102 - Lê Quang Phúc
    - 23127212 - Nguyễn Quang Đăng Khoa
    - 23127241 - Đoàn Thành Phát
    - 23127332 - Trần Tiến Cường
    - 23127442 - Trầm Hữu Nhân


# **Đánh giá baseline đối với mô hình Tesseract5x**
Mục tiêu của notebook này là đánh giá hiệu quả của mô hình `Tesseract5x` trong bài toán nhận diện ký tự quang học (OCR) trên bộ dữ liệu đã chuẩn bị. Kết quả đánh giá sẽ giúp so sánh chất lượng giữa các mô hình OCR khác nhau.

## 1. Cài đặt thư viện và môi trường 

In [ ]:
%%bash
# Cài đặt Tesseract 5.x vào hệ điều hành
add-apt-repository ppa:alex-p/tesseract-ocr5 -y
apt-get update -qq
apt-get install -y -qq tesseract-ocr

# Tải model tiếng Việt
wget https://github.com/tesseract-ocr/tessdata/raw/main/script/Vietnamese.traineddata -O /usr/share/tesseract-ocr/5/tessdata/Vietnamese.traineddata

pip install -q pytesseract Levenshtein pandas tqdm

### 1.1 Mount tới Google Drive để lấy dataset

Do tập dữ liệu có kích thước lớn, việc đọc trực tiếp từ **Google Drive** có thể gây ra hiện tượng thắt nút cổ chai băng thông làm chậm đáng kể quá trình suy luận của mô hình. Nên là:
- **Mount Google Drive** để lấy file nén `processed_data.zip`.
- **Giải nén trực tiếp vào bộ nhớ cục bộ** của máy ảo Colab (`/content/local_data`). Thao tác này giúp thao tác đọc ảnh trong vòng lặp đánh giá sau này đạt tốc độ tối đa.

In [ ]:
import os
import json
import unicodedata
import pandas as pd
import Levenshtein
import pytesseract

from pathlib import Path
from tqdm import tqdm
from PIL import Image
from google.colab import drive

# Kết nối Google Drive
drive.mount('/content/drive')

# Đường dẫn
ZIP_PATH = Path('/content/drive/MyDrive/IntroToML - OCR - data/processed_data.zip')
LOCAL_ROOT = Path('/content/local_data')

# Giải nén
if ZIP_PATH.exists():
    if not LOCAL_ROOT.exists():
        !unzip -q "{ZIP_PATH}" -d "{LOCAL_ROOT}"
    else:
        print("Dữ liệu đã có sẵn.")
else:
    print(f"❌ LỖI: Không tìm thấy file {ZIP_PATH}")

TEST_DIR = LOCAL_ROOT / 'test'
print(f"\nĐường dẫn thư mục Test: {TEST_DIR}")

Mounted at /content/drive

 Đường dẫn thư mục Test: /content/local_data/test


## 2. Các hàm hỗ trợ
Trước khi đưa dữ liệu vào mô hình, ta cần chuẩn hóa văn bản. Trong tiếng Việt, hiện tượng khác biệt bảng mã `Unicode` rất phổ biến và có thể dẫn đến việc đánh giá sai lệch dù mặt chữ giống nhau. 
Hàm `normalize_text` sử dụng chuẩn **NFC** để đưa toàn bộ văn bản về một dạng mã hóa thống nhất, đồng thời chuyển về chữ thường để đánh giá tập trung vào mặt chữ thay vì in hoa/in thường.

Để đo lường hiệu suất của mô hình OCR, độ đo **CER (Character error rate - Tỉ lệ lỗi ký tự)** được sử dụng. CER dựa trên khoảng cách `Levenshtein`, đếm số lượng thao tác tối thiểu cần thiết để biến đổi chuỗi dự đoán thành chuỗi thực tế (ground truth).

Công thức tính toán:
$$CER = \frac{S + D + I}{N}$$

Trong đó:
* $S$ (Substitutions): Số ký tự bị thay thế sai.
* $D$ (Deletions): Số ký tự bị bỏ sót.
* $I$ (Insertions): Số ký tự bị chèn thừa.
* $N$: Tổng số ký tự của nhãn gốc (ground truth).

*Lưu ý:* Giá trị CER càng gần 0.0 thì mô hình nhận dạng càng chính xác.

In [3]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    return text.strip().lower()

def calculate_cer(pred: str, gt: str) -> float:
    pred = normalize_text(pred)
    gt = normalize_text(gt)

    if len(gt) == 0:
        return 1.0 if len(pred) > 0 else 0.0

    edit_dist = Levenshtein.distance(pred, gt)
    cer = edit_dist / len(gt)
    return cer

## 3. Khởi tạo và đánh giá mô hình

Mô hình **Tesseract5x** được khởi tạo với trọng số dành riêng cho tiếng Việt (`TESSERACT_LANG='Vietnames'`) và sự lựa chọn thuật toán **page segmentation mode (PSM)** hợp lý – cơ chế phân tách các khối văn bản trên ảnh trước khi nhận dạng. Để tối ưu hóa, pipeline này triển khai một **cơ chế lựa chọn PSM** dựa trên tỷ lệ khung hình của ảnh đầu vào:
* **$Aspect ratio > 2.5$ (Ảnh dài và hẹp ngang):** Mô hình giả định đây là một dòng văn bản đơn lẻ. Cấu hình `--psm 7` được áp dụng.
* **$Aspect ratio \le 2.5$ (Ảnh có dạng khối vuông/chữ nhật đều):** Mô hình giả định đây là một đoạn văn bản nhiều dòng. Cấu hình `--psm 6` được áp dụng.

**Quá trình đánh giá:**
- Sử dụng `PIL.Image` để tải ảnh và tính toán kích thước.
- Áp dụng luật Aspect Ratio để quyết định cấu hình `--psm`.
- Truyền ảnh qua engine Tesseract để lấy kết quả dạng chuỗi (string).
- Tính toán điểm CER cho từng ảnh, **chỉ lưu lại log của những ảnh có $CER > 0$** (những ảnh dự đoán sai). Việc này giúp tối ưu hóa bộ nhớ và tập trung hoàn toàn vào việc phân tích lỗi  ở bước sau.

In [5]:
# Cấu hình Tesseract
print("Đang nạp mô hình Teseract...")
TESSERACT_LANG = "Vietnamese"

# Quét qua tất cả thư mục con
all_test_samples = []
subfolders = [f for f in TEST_DIR.iterdir() if f.is_dir()]

for subfolder in subfolders:
    label_file = subfolder / 'label.json'
    if not label_file.exists():
        continue

    with open(label_file, 'r', encoding='utf-8') as f:
        ground_truths = json.load(f)

    for img_name, gt_text in ground_truths.items():
        img_path = subfolder / img_name
        if img_path.exists():
            all_test_samples.append((img_path, gt_text))

print(f"Thực hiện đánh giá trên {len(all_test_samples)} ảnh \n")

# Vòng lặp đánh giá
total_cer = 0.0
error_logs = []

for img_path, gt_text in tqdm(all_test_samples, desc="Đang đánh giá"):
    try:
        img = Image.open(str(img_path))

        width, height = img.size
        aspect_ratio = width / height

        # Dòng
        if aspect_ratio > 2.5:
            config_mode = r'--psm 7'
        else:
            # Đoạn
            config_mode = r'--psm 6'

        # Suy luận bằng Tesseract
        pred_text = pytesseract.image_to_string(img, lang=TESSERACT_LANG, config=config_mode)
    except Exception as e:
        pred_text = ""

    # Tính điểm
    cer_score = calculate_cer(pred_text, gt_text)
    total_cer += cer_score

    # Ghi log ảnh lỗi
    if cer_score > 0:
        error_logs.append({
            "folder": img_path.parent.name,
            "image": img_path.name,
            "ground_truth": normalize_text(gt_text),
            "prediction": normalize_text(pred_text),
            "cer_score": round(cer_score, 4)
        })

# Tính toán CER trung bình
test_samples = len(all_test_samples)
average_cer = total_cer / test_samples if test_samples > 0 else 0

print(f"\nCER trung bình: {average_cer * 100:.2f} %")
print(f"Số lượng ảnh dự đoán sai: {len(error_logs)}")

Đang nạp mô hình Teseract...
Thực hiện đánh giá trên 15000 ảnh 



Đang đánh giá: 100%|██████████| 15000/15000 [40:23<00:00,  6.19it/s]


CER trung bình: 98.59 %
Số lượng ảnh dự đoán sai: 14741


## 4. Lưu trữ
Sau khi hoàn thành quá trình đánh giá, tạo ra file CSV `baseline_Tesseract_report.csv` để báo cáo các lỗi và được lưu trên Google Drive, phục vụ cho quá trình phân tích, đối chiếu khi tinh chỉnh các mô hình sau này.

In [ ]:
# Chuyển đổi danh sách lỗi thành Pandas DataFrame
df_errors = pd.DataFrame(error_logs)

# Sắp xếp từ lỗi nặng nhất xuống nhẹ nhất
df_errors = df_errors.sort_values(by="cer_score", ascending=False)

# Lưu file nội bộ
report_path = LOCAL_ROOT / 'baseline_Tesseract_report.csv'
df_errors.to_csv(report_path, index=False, encoding='utf-8-sig')

# Copy sang Google Drive
drive_report_path = Path('/content/drive/MyDrive/IntroToML - OCR - data/baseline_Tesseract_report.csv')
!cp "{report_path}" "{drive_report_path}"

print(f"Đã lưu lại report của baseline_Tesseract")

Đã lưu lại report của baseline_Tesseract
